In [ ]:
import copy
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split, Subset, ConcatDataset
from torchvision import transforms
from torchvision.datasets import MNIST, EMNIST

# ============================================================
# 1. REPRODUCIBILITY & DEVICE
# ============================================================
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available() 
    else "mps" if torch.backends.mps.is_available() 
    else "cpu"
)

# ============================================================
# 2. DATA PREPARATION
# ============================================================
# Domain-specific statistics to prevent validation normalization leaks
mnist_normalize = transforms.Normalize((0.1307,), (0.3081,))
emnist_normalize = transforms.Normalize((0.1722,), (0.3309,))

mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    mnist_normalize
])

emnist_transform = transforms.Compose([
    lambda img: transforms.functional.rotate(img, -90),
    lambda img: transforms.functional.hflip(img),
    transforms.ToTensor(),
    emnist_normalize
])

emnist_target_transform = lambda y: y - 1 if y > 0 else y

# --- Load MNIST Source Dataset ---
mnist_full = MNIST(root="data", train=True, transform=mnist_transform, download=True)
mnist_test = MNIST(root="data", train=False, transform=mnist_transform, download=True)

mnist_generator = torch.Generator().manual_seed(42)
mnist_train, mnist_val = random_split(mnist_full, [55000, 5000], generator=mnist_generator)

# --- Load EMNIST Target Dataset ---
fraction = 0.05
emnist_full_train = EMNIST(
    root="data", 
    split="letters", 
    train=True, 
    transform=emnist_transform,
    target_transform=emnist_target_transform,
    download=True
)

emnist_test = EMNIST(
    root="data", 
    split="letters", 
    train=False, 
    transform=emnist_transform,
    target_transform=emnist_target_transform,
    download=True
)

# Create a small subset to simulate low-data transfer regime
subset_size = int(len(emnist_full_train) * fraction)
subset_generator = torch.Generator().manual_seed(42)
indices = torch.randperm(len(emnist_full_train), generator=subset_generator)[:subset_size]
emnist_subset = Subset(emnist_full_train, indices)

train_size = int(0.8 * subset_size)
val_size = subset_size - train_size

# Decoupled seed (43) to prevent structural index matching across domains
emnist_generator = torch.Generator().manual_seed(43)
emnist_train, emnist_val = random_split(
    emnist_subset,
    [train_size, val_size],
    generator=emnist_generator
)

# --- Setup DataLoaders ---
batch_size = 128
loader_args = {"batch_size": batch_size, "num_workers": 0}

mnist_train_loader = DataLoader(mnist_train, shuffle=True, **loader_args)
mnist_val_loader   = DataLoader(mnist_val, shuffle=False, **loader_args)

emnist_train_loader = DataLoader(emnist_train, shuffle=True, **loader_args)
emnist_val_loader   = DataLoader(emnist_val, shuffle=False, **loader_args)

# ============================================================
# 3. MODEL (ENCODER + CLASSIFIER HEAD)
# ============================================================
class Classifier(nn.Module):
    def __init__(self, latent_dim=64, num_classes=26):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, latent_dim)
        )

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.classifier(z)

# ============================================================
# 4. UTILITIES
# ============================================================
class EarlyStopping:
    def __init__(self, model, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_weights = copy.deepcopy(model.state_dict())

    def __call__(self, model, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
            return False

        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)

# ============================================================
# 5. TRAINING ENGINE
# ============================================================
def plot_history(history, title="Training History"):
    epochs = range(1, len(history["train_loss"]) + 1)
    plt.figure(figsize=(14, 4))

    # --- Loss Plot ---
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title} - Loss")
    plt.legend()

    # --- Accuracy Plot ---
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="Train Accuracy")
    plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title} - Accuracy")
    plt.legend()

    plt.tight_layout()
    plt.show()

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    return total_loss / len(loader), correct / total

def run_experiment(
    model, train_loader, val_loader, epochs=20, freeze_encoder=False,
    unfreeze_epoch=None, encoder_lr=1e-3, classifier_lr=1e-3, fine_tune_lr=1e-4
):
    criterion = nn.CrossEntropyLoss()

    if freeze_encoder:
        for param in model.encoder.parameters():
            param.requires_grad = False

    optimizer = torch.optim.Adam([
        {"params": model.encoder.parameters(), "lr": encoder_lr},
        {"params": model.classifier.parameters(), "lr": classifier_lr}
    ])

    model.to(device)
    es = EarlyStopping(model, patience=5)
    
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(epochs):
        # Scheduled Fine-Tuning Step
        if unfreeze_epoch is not None and epoch == unfreeze_epoch:
            print(f"\n>>> Epoch {epoch+1}: Unfreezing encoder")
            for param in model.encoder.parameters():
                param.requires_grad = True
            optimizer.param_groups[0]["lr"] = fine_tune_lr

        # Training Phase
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += y.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total

        # Validation Phase
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch+1:02d} | Train Loss {train_loss:.4f} | Val Loss {val_loss:.4f} | Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

        if es(model, val_loss):
            print("Early stopping triggered.")
            break

    es.restore(model)
    return history

# ============================================================
# 6. VALIDATION SWEEP STAGE
# ============================================================
latent_dim = 64

print(f"\n" + "="*60)
print(f"RUNNING EXPERIMENTS FOR LATENT_DIM = {latent_dim}")
print("="*60)

# --- Step 1: Source Domain Training (MNIST) ---
print(f"\n--- STEP 1: MNIST PRETRAINING ---")
mnist_model = Classifier(latent_dim=latent_dim, num_classes=10)
mnist_hist = run_experiment(
    model=mnist_model, train_loader=mnist_train_loader, val_loader=mnist_val_loader, epochs=20
)
pretrained_encoder = copy.deepcopy(mnist_model.encoder.state_dict())

# --- Step 2: Target Domain Evaluation with Transfer Learning ---
print(f"\n--- STEP 2: TRANSFER LEARNING ---")
transfer_model = Classifier(latent_dim=latent_dim, num_classes=26)
transfer_model.encoder.load_state_dict(pretrained_encoder)

transfer_hist = run_experiment(
    model=transfer_model, train_loader=emnist_train_loader, val_loader=emnist_val_loader,
    epochs=40, freeze_encoder=True, unfreeze_epoch=5, fine_tune_lr=1e-4
)

# --- Step 3: Target Domain Evaluation from Scratch ---
print(f"\n--- STEP 3: TRAINING FROM SCRATCH ---")
scratch_model = Classifier(latent_dim=latent_dim, num_classes=26)
scratch_hist = run_experiment(
    model=scratch_model, train_loader=emnist_train_loader, val_loader=emnist_val_loader, epochs=40
)

# --- Strategy Evaluation ---
plot_history(transfer_hist, title=f"Transfer Learning (latent_dim={latent_dim})")
plot_history(scratch_hist, title=f"Training From Scratch (latent_dim={latent_dim})")

transfer_best_acc = max(transfer_hist["val_acc"])
scratch_best_acc = max(scratch_hist["val_acc"])

if transfer_best_acc > scratch_best_acc:
    best_strategy = "Transfer"
    winning_hist = transfer_hist
    best_val_acc = transfer_best_acc
else:
    best_strategy = "Scratch"
    winning_hist = scratch_hist
    best_val_acc = scratch_best_acc

print("\n" + "="*50)
print(f"BEST STRATEGY: {best_strategy}")
print(f"BEST VALIDATION ACCURACY: {best_val_acc*100:.2f}%")
print("="*50)

# ============================================================
# 7. PRODUCTION RETRAINING STAGE (FULL TRAIN + VAL SUBSET)
# ============================================================
print(f"\n--- RETRAINING BEST MODEL ON FULL SUBSET ---")

full_emnist_subset = ConcatDataset([emnist_train, emnist_val])
full_subset_loader = DataLoader(full_emnist_subset, shuffle=True, **loader_args)

# Find optimal historical epoch length based on peak performance index
optimal_epochs = np.argmax(winning_hist["val_acc"]) + 1
print(f"Optimal training length determined from historical validation curve: {optimal_epochs} epochs")

champion_model = Classifier(latent_dim=latent_dim, num_classes=26)
criterion = nn.CrossEntropyLoss()

if best_strategy == "Transfer":
    champion_model.encoder.load_state_dict(pretrained_encoder)
    for param in champion_model.encoder.parameters():
        param.requires_grad = False

    optimizer = torch.optim.Adam([
        {"params": champion_model.encoder.parameters(), "lr": 0.0},
        {"params": champion_model.classifier.parameters(), "lr": 1e-3}
    ])
    unfreeze_epoch = 5
    fine_tune_lr = 1e-4
else:
    optimizer = torch.optim.Adam([
        {"params": champion_model.encoder.parameters(), "lr": 1e-3},
        {"params": champion_model.classifier.parameters(), "lr": 1e-3}
    ])
    unfreeze_epoch = None

champion_model.to(device)

for epoch in range(optimal_epochs):
    if unfreeze_epoch is not None and epoch == unfreeze_epoch:
        print(f">>> Epoch {epoch+1}: Fine-tuning encoder...")
        for param in champion_model.encoder.parameters():
            param.requires_grad = True
        optimizer.param_groups[0]["lr"] = fine_tune_lr

    champion_model.train()
    total_loss = 0.0
    for x, y in full_subset_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        logits = champion_model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Retrain Epoch {epoch+1:02d}/{optimal_epochs:02d} | Loss: {total_loss / len(full_subset_loader):.4f}")

# ============================================================
# 8. FINAL HOLD-OUT TEST SET EVALUATION
# ============================================================
champion_model.eval()
emnist_test_loader = DataLoader(emnist_test, shuffle=False, **loader_args)

test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for x, y in emnist_test_loader:
        x, y = x.to(device), y.to(device)
        logits = champion_model(x)
        loss = criterion(logits, y)
        
        test_loss += loss.item()
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)

avg_test_loss = test_loss / len(emnist_test_loader)
test_acc = correct / total

print("\n" + "="*50)
print("FINAL TEST SET BENCHMARK SUMMARY (POST-RETRAINING)")
print("="*50)
print(f"Architecture Config -> Dim: {latent_dim:2d} | Strategy: {best_strategy}")
print(f"Final Test Loss         -> {avg_test_loss:.4f}")
print(f"Final Test Accuracy     -> {test_acc*100:.2f}%")
print("="*50)